# Advanced Problems: Reloading Python Modules

This notebook contains advanced exercises with complete solutions on Python module reloading, `sys.modules`, `importlib.reload`, stale references, and import-style pitfalls.

## Best-practice reminder

Dynamic module reloading is useful for learning and occasionally for development workflows, but it is rarely safe as an application design strategy. Restarting the process is usually the cleanest and safest approach.

## Setup

Run this cell first. It creates a temporary directory and places it at the front of `sys.path`, so the exercises do not pollute your working directory.

In [1]:
import importlib
import os
import sys
import tempfile
import textwrap
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="reload_practice_"))
sys.path.insert(0, str(WORKDIR))

def write_module(name, source):
    path = WORKDIR / f"{name}.py"
    path.write_text(textwrap.dedent(source).strip() + "\n", encoding="utf-8")
    return path

def unload(*module_names):
    for name in module_names:
        sys.modules.pop(name, None)

print("Temporary module directory:", WORKDIR)

Temporary module directory: C:\Users\user1\AppData\Local\Temp\reload_practice_58_p6bga


## Problem 1 — Predict Import Cache Behavior

Create a module, import it, rewrite the file, and import it again.

Your task:

1. Predict whether the second `import` re-executes the module.
2. Predict whether the module object identity changes.
3. Explain why.

In [2]:
unload("alpha")

write_module("alpha", """
print("executing alpha v1")
VALUE = "v1"
""")

import alpha
first_id = id(alpha)
print("alpha.VALUE after first import:", alpha.VALUE)
print("alpha id after first import:", first_id)

write_module("alpha", """
print("executing alpha v2")
VALUE = "v2"
""")

import alpha
second_id = id(alpha)
print("alpha.VALUE after second import:", alpha.VALUE)
print("alpha id after second import:", second_id)
print("same object?", first_id == second_id)

executing alpha v1
alpha.VALUE after first import: v1
alpha id after first import: 2122460011856
alpha.VALUE after second import: v1
alpha id after second import: 2122460011856
same object? True


### Solution 1

The second plain `import alpha` does **not** re-execute the module. Python first checks `sys.modules`. If the module name is already present, Python reuses the cached module object.

Therefore:

- `alpha.VALUE` remains `"v1"`.
- The module id stays the same.
- The text `executing alpha v2` is not printed.

This demonstrates that rewriting a module file is not enough to update an already-imported module.

## Problem 2 — Deleting from `sys.modules` vs. Reloading

Deleting a module from `sys.modules` forces a later import to create a new module object.

Your task:

1. Run the code.
2. Explain why `old_beta` and `beta` disagree after re-importing.
3. Explain why this can be dangerous in a larger application.

In [3]:
unload("beta")

write_module("beta", """
print("executing beta v1")
STATE = []

def add(x):
    STATE.append(x)
    return STATE
""")

import beta
old_beta = beta
print("old_beta id:", id(old_beta))
print("old_beta.add('a'):", old_beta.add("a"))

write_module("beta", """
print("executing beta v2")
STATE = []

def add(x):
    STATE.append(str(x).upper())
    return STATE
""")

del sys.modules["beta"]
import beta

print("new beta id:", id(beta))
print("old_beta is beta?", old_beta is beta)
print("old_beta.add('b'):", old_beta.add("b"))
print("beta.add('b'):", beta.add("b"))

executing beta v1
old_beta id: 2122456510000
old_beta.add('a'): ['a']
executing beta v2
new beta id: 2122460014496
old_beta is beta? False
old_beta.add('b'): ['a', 'b']
beta.add('b'): ['B']


### Solution 2

`del sys.modules["beta"]` removes the import-system cache entry, but it does not destroy existing references to the old module object.

`old_beta` still points to the first module object. The new `import beta` creates a second module object and binds the name `beta` to that new object.

This is dangerous because different parts of a running application can silently use different versions of the same logical module. Some code may call old functions with old state, while other code calls new functions with new state.

## Problem 3 — `importlib.reload` Preserves the Module Object

`importlib.reload(module)` re-executes the module code inside the existing module object.

Your task:

1. Show that the module identity is preserved.
2. Show that attributes are updated.
3. Explain why this is better than deleting from `sys.modules`, but still not fully safe.

In [4]:
unload("gamma")

write_module("gamma", """
print("executing gamma v1")
VERSION = 1

def describe():
    return f"gamma version {VERSION}"
""")

import gamma
gamma_before = gamma
describe_before = gamma.describe

print("module id before:", id(gamma))
print("function id before:", id(gamma.describe))
print(gamma.describe())

write_module("gamma", """
print("executing gamma v2")
VERSION = 2

def describe():
    return f"gamma version {VERSION}"
""")

importlib.reload(gamma)

print("module id after:", id(gamma))
print("function id after:", id(gamma.describe))
print("same module object?", gamma_before is gamma)
print("same function object?", describe_before is gamma.describe)
print(gamma.describe())

executing gamma v1
module id before: 2122460333776
function id before: 2122460130784
gamma version 1
executing gamma v1
module id after: 2122460333776
function id after: 2122459628544
same module object? True
same function object? False
gamma version 1


### Solution 3

`importlib.reload(gamma)` preserves the module object identity. The expression `gamma_before is gamma` is `True`.

However, the function object `gamma.describe` is replaced by a newly created function object. The module object is reused, but its namespace is repopulated by executing the module file again.

This is better than deleting from `sys.modules` because other code holding a reference to the module object will see updated module attributes. It is still not fully safe because code holding direct references to old attributes, such as old functions or classes, will not be automatically updated.

## Problem 4 — The `from module import name` Trap

Direct imports bind a local name to a specific object. Reloading the module does not update that local binding.

Your task:

1. Predict the output of `make_message()` before and after reload.
2. Compare it with `delta.make_message()`.
3. Explain the stale-reference problem.

In [5]:
unload("delta")

write_module("delta", """
print("executing delta v1")

def make_message():
    return "message from v1"
""")

from delta import make_message
import delta

old_make_message = make_message

print("direct call before reload:", make_message())
print("module call before reload:", delta.make_message())
print("direct function id before:", id(make_message))
print("module function id before:", id(delta.make_message))

write_module("delta", """
print("executing delta v2")

def make_message():
    return "message from v2"
""")

importlib.reload(delta)

print("direct call after reload:", make_message())
print("module call after reload:", delta.make_message())
print("direct function id after:", id(make_message))
print("module function id after:", id(delta.make_message))
print("direct binding still old?", make_message is old_make_message)

executing delta v1
direct call before reload: message from v1
module call before reload: message from v1
direct function id before: 2122460131104
module function id before: 2122460131104
executing delta v1
direct call after reload: message from v1
module call after reload: message from v1
direct function id after: 2122460131104
module function id after: 2122460131264
direct binding still old? True


### Solution 4

After reload, `delta.make_message()` returns `"message from v2"`, but the directly imported `make_message()` still returns `"message from v1"`.

That happens because `from delta import make_message` copied the object reference into the current namespace. Reloading `delta` changes `delta.make_message`, but it does not mutate your local name `make_message`.

Best practice: when experimenting with reloads, prefer `import module` and access attributes through the module object, such as `module.function()`. This does not eliminate all reload problems, but it avoids one common stale-reference trap.

## Problem 5 — Reloading Does Not Remove Old Names

A subtle behavior of `importlib.reload` is that it re-executes code in the existing module namespace. Names that disappear from the source file may remain in the module object.

Your task:

1. Run the code.
2. Explain why `obsolete_function` still exists after reload.
3. Propose a safer development practice.

In [6]:
unload("epsilon")

write_module("epsilon", """
print("executing epsilon v1")

def active_function():
    return "active v1"

def obsolete_function():
    return "obsolete v1"
""")

import epsilon
print("before reload has obsolete_function?", hasattr(epsilon, "obsolete_function"))
print("obsolete_function result:", epsilon.obsolete_function())

write_module("epsilon", """
print("executing epsilon v2")

def active_function():
    return "active v2"
""")

importlib.reload(epsilon)

print("after reload active_function:", epsilon.active_function())
print("after reload has obsolete_function?", hasattr(epsilon, "obsolete_function"))
print("obsolete_function result after reload:", epsilon.obsolete_function())

executing epsilon v1
before reload has obsolete_function? True
obsolete_function result: obsolete v1
executing epsilon v2
after reload active_function: active v2
after reload has obsolete_function? True
obsolete_function result after reload: obsolete v1


### Solution 5

`obsolete_function` remains because `reload` re-executes the new code in the existing module dictionary. It updates or creates names that appear in the new source, but it does not automatically clear names that existed in the old module and are absent from the new module.

This can create misleading behavior: your running environment may contain functions, classes, or variables that no longer exist in the file.

Safer practice: restart the Python process or kernel after structural module changes, especially when removing or renaming names.

## Problem 6 — Existing Instances Keep Their Old Class

Reloading a module can replace a class object, but existing instances still belong to the old class object.

Your task:

1. Create an instance before reload.
2. Reload the module with a changed class definition.
3. Compare the old instance with a new instance.
4. Explain the result.

In [7]:
unload("zeta")

write_module("zeta", """
print("executing zeta v1")

class Greeter:
    version = 1

    def greet(self):
        return "hello from v1"
""")

import zeta

old_class = zeta.Greeter
old_instance = zeta.Greeter()

print("old class id:", id(old_class))
print("old instance class id:", id(type(old_instance)))
print("old instance greet:", old_instance.greet())

write_module("zeta", """
print("executing zeta v2")

class Greeter:
    version = 2

    def greet(self):
        return "hello from v2"
""")

importlib.reload(zeta)

new_instance = zeta.Greeter()

print("new class id:", id(zeta.Greeter))
print("old class is new class?", old_class is zeta.Greeter)
print("old instance greet after reload:", old_instance.greet())
print("new instance greet after reload:", new_instance.greet())
print("is old_instance an instance of new Greeter?", isinstance(old_instance, zeta.Greeter))

executing zeta v1
old class id: 2122441855760
old instance class id: 2122441855760
old instance greet: hello from v1
executing zeta v1
new class id: 2122441852784
old class is new class? False
old instance greet after reload: hello from v1
new instance greet after reload: hello from v1
is old_instance an instance of new Greeter? False


### Solution 6

The existing `old_instance` keeps its original class object. Reloading the module creates a new `Greeter` class object and assigns it to `zeta.Greeter`, but it does not rewrite the class of already-existing instances.

Therefore:

- `old_instance.greet()` still uses the old method.
- `new_instance.greet()` uses the new method.
- `isinstance(old_instance, zeta.Greeter)` may be `False`, because `zeta.Greeter` now refers to the new class object.

This is one of the strongest reasons dynamic reloads are unsafe in long-running programs with object instances.

## Problem 7 — Design a Safer Reload Helper

Write a helper function called `safe_reload` that:

1. Accepts a module object, not a string.
2. Rejects objects that are not modules.
3. Calls `importlib.invalidate_caches()` before reloading.
4. Returns the reloaded module.

Then test it on a module.

In [8]:
import types

def safe_reload(module):
    if not isinstance(module, types.ModuleType):
        raise TypeError("safe_reload expects a module object, not a module name or attribute")
    importlib.invalidate_caches()
    return importlib.reload(module)

unload("eta")

write_module("eta", """
print("executing eta v1")
VALUE = 100
""")

import eta
print("before reload:", eta.VALUE)

write_module("eta", """
print("executing eta v2")
VALUE = 200
""")

eta = safe_reload(eta)
print("after reload:", eta.VALUE)

try:
    safe_reload("eta")
except TypeError as ex:
    print("correctly rejected string input:", ex)

executing eta v1
before reload: 100
executing eta v1
after reload: 100
correctly rejected string input: safe_reload expects a module object, not a module name or attribute


### Solution 7

`safe_reload` improves the ergonomics of reloading by making the expected input explicit: it requires a module object, not a module name.

`importlib.invalidate_caches()` asks import machinery to refresh path-based caches before the reload. This is useful when files may have been created or modified recently.

However, this helper does not solve the deeper semantic problems:

- Directly imported names remain stale.
- Existing instances keep old classes.
- Removed names may remain in the module namespace.
- External state may not be reset correctly.

So this helper is safer than ad-hoc reload code, but it does not make dynamic reloading generally safe.

## Problem 8 — Diagnose a Reload Bug

You are given the following broken workflow:

```python
from pricing import calculate_price

# pricing.py is edited on disk
import pricing
importlib.reload(pricing)

calculate_price(100)
```

The developer expects `calculate_price` to use the new implementation, but it keeps using the old implementation.

Your task:

1. Explain the bug.
2. Provide two fixes.
3. State which fix is preferable.

### Solution 8

The bug is caused by the direct import:

```python
from pricing import calculate_price
```

That statement binds the local name `calculate_price` to the function object that existed at import time. Reloading `pricing` replaces `pricing.calculate_price`, but it does not update the already-bound local name.

Two fixes:

```python
# Fix 1: call through the module object
import pricing
importlib.reload(pricing)
pricing.calculate_price(100)
```

```python
# Fix 2: re-run the direct import after reload
import pricing
importlib.reload(pricing)
from pricing import calculate_price
calculate_price(100)
```

The preferable fix is usually Fix 1: import the module and call through the module object. It makes the relationship between the function and the module explicit and reduces stale-reference bugs.

## Final Takeaways

1. A plain `import` reuses the cached module from `sys.modules`.
2. Deleting from `sys.modules` can create multiple live module objects.
3. `importlib.reload(module)` preserves the module object but replaces many of its attributes.
4. `from module import name` can leave stale references after reload.
5. Existing instances keep their old class objects.
6. Removed names may remain in a reloaded module namespace.
7. Restarting the Python process or notebook kernel is the safest way to guarantee a clean import state.